# 🔄 数据透视表 (Pivot Table) 与 行列转换

> **灵魂拷问：它和 `groupby` 有啥区别？**
> *   `groupby`（分组）：用来给**机器看**的。它把数据聚合成“长长的清单”（长表），适合做进一步的计算或画图。
> *   `pivot_table`（透视表）：用来给**老板看**的。它能把数据变成“带行和列的二维网格”（宽表），像极了 Excel 里的透视表！

---
## 1. 准备示例数据：电脑各季度的销量

In [2]:
import pandas as pd
import numpy as np

# 我们伪造了一份常见的工作数据 (长表格式，每天/每笔订单记一行)
data = {
    '品牌': ['联想', '联想', '戴尔', '戴尔', '惠普', '联想', '戴尔', '惠普'],
    '季度': ['Q1', 'Q2', 'Q1', 'Q2', 'Q1', 'Q1', 'Q2', 'Q2'],
    '定位': ['高端', '高端', '普通', '高端', '普通', '普通', '普通', '高端'],
    '销量': [120, 150, 80, 110, 90, 200, 100, 130]
}
df = pd.DataFrame(data)
display(df)

,品牌,季度,定位,销量
0,联想,Q1,高端,120
1,联想,Q2,高端,150
2,戴尔,Q1,普通,80
3,戴尔,Q2,高端,110
4,惠普,Q1,普通,90
5,联想,Q1,普通,200
6,戴尔,Q2,普通,100
7,惠普,Q2,高端,130


---
## 2. 同样的任务，`groupby` 会做成什么样？ (长表)
假设老板问：**“每个品牌、每个季度的总销量是多少？”**

In [4]:
df_result1 = df.groupby(['品牌','季度']).sum('销量').reset_index()
display(df_result1)

,品牌,季度,销量
0,惠普,Q1,90
1,惠普,Q2,130
2,戴尔,Q1,80
3,戴尔,Q2,210
4,联想,Q1,320
5,联想,Q2,150


---
## 3. 换成 `pivot_table` 做成网格结构 (宽表)
老板说：“我不喜欢上下对应，我要左边是品牌，上面是季度，中间是销量！还要给我标出总计！”

In [6]:
# 注意：这里你用的是 `pivot`，这是一个基础版转换工具（只能重塑，不能计算）。
# 遇到重复数据时，普通的 pivot 会报错！
df_result2 = df_result1.pivot(index='品牌', columns='季度', values='销量')
display("基础版 pivot (不能做去重运算，无总计功能):")
display(df_result2)

'基础版 pivot (不能做去重运算，无总计功能):'

季度,Q1,Q2
品牌,,
惠普,90,130
戴尔,80,210
联想,320,150


---
## 👑 4. 终极一招：为什么不直接用 `pivot`，而是要用 `pivot_table`？

你在上面用的是 `df.pivot()`。这是一个**低配版**。
如果一个员工在同一个季度卖了两次“联想”，用普通的 `pivot` 程序会直接崩溃！

所以，高级玩家永远只用 **`pd.pivot_table()`**！
它自带 **一键汇总 (margins=True)** 和 **内置聚合求和 (aggfunc='sum')**，这就是我说的做给老板看的终极一招。

In [8]:
# ⚠️ 神级工具：pd.pivot_table() 
# 注意它是 pd. 调用的，而不是直接用 df.

df_ultimate = pd.pivot_table(
    df,
    index='品牌',         # 左边的列
    columns='季度',       # 顶部的列
    values='销量',        # 中间的数字
    aggfunc='sum',        # ⭐ 这个最牛，如果在同一季度卖
    fill_value=0,
    margins=True          # ⭐ 终极一招：一键在最右侧和最底部加上【总计All】         
)

display("这是带总计、自动去重合并的终极 pivot_table 版:")
display(df_ultimate)

'这是带总计、自动去重合并的终极 pivot_table 版:'

季度,Q1,Q2,All
品牌,,,
惠普,90,130,220
戴尔,80,210,290
联想,320,150,470
All,490,490,980


---
## 🚀 5. 进阶玩法：多层透视与多重计算

刚刚你掌握的是 1 维对应 1 维。如果老板问：
**“我想看每个品牌下，不同‘定位’（高端/普通）在各季度的销量和平均销量分布！”**

很简单，`index`, `columns`, `values`, `aggfunc` 都可以传**列表 (List)**。

In [ ]:
# 进阶多层透视
df_multi = pd.pivot_table(
    df,
    index=['品牌', '定位'],               # 变成多层级左侧标签
    columns='季度',
    values='销量',
    aggfunc=['sum', 'mean'],            # ⭐ 一次性算多种指标：求和 ＋ 平均
    fill_value=0
)

display("这是多维度、多计算的透视表:")
display(df_multi)
# 思考题：跑出来之后，仔细看看表头的层级结构 (MultiIndex)。

---
## 🧊 6. 【高阶武功反制】从宽表退回长表：`melt`

如果你从外面（比如政府网站或者老旧的Excel）拿到了一个**横向铺开**的数据（别人已经用 pivot 铺好了），现在你要把它倒腾回数据库，你需要把它**“融化（melt）”成长表**。

> **大白话**：把那些表示“季度”、“月份”的列名，通通拍扁成一列叫做“时间”的数据。

In [ ]:
# 假设我们拿到了刚才处理好的普通的 df_result2 (宽表), 我们把它重置一下变成干净的表
df_wide = df_result2.reset_index()
df_wide = df_wide.rename_axis(None, axis=1) # 去掉隐形的列名标签
display("1. 假设这是你从Excel拿到的一张宽表：", df_wide)

# 开始融化操作！
df_melted = pd.melt(
    df_wide, 
    id_vars=['品牌'],              # ⭐ 这列是"雷打不动"的，保持原样 (当做锚点)
    value_vars=['Q1', 'Q2'],     # 这些是被铺开的列，我要把它们“拍扁”
    var_name='季度',              # 拍扁后，原来做列名的那些词，起个什么是头？(如“季度”)
    value_name='销售额'           # 对应格子里的那些数字，起个什么叫头？(如“销售额”)
)

display("2. 融化后，重新变回可以写入数据库的长表：", df_melted)